In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

: 

In [ ]:
rv_file = pd.read_csv('/Users/marta/Documents/LunaNetSimulatorV2/output/ExampleEndurance/rv_moon_pa.csv', names = ['Time', 'x', 'y', 'z', 'vx', 'vy', 'vz'])
rover_file = pd.read_csv('/Users/marta/Documents/LunaNetSimulatorV2/output/ExampleEndurance/rover_vec_pa.csv', names = ['Time', 'x', 'y', 'z', 'vx', 'vy', 'vz'])
pseudorange_file = pd.read_csv('/Users/marta/Documents/LunaNetSimulatorV2/output/ExampleEndurance/z_pr.csv', names = ['Time', 'rho'])
pseudorange_rate_file = pd.read_csv('/Users/marta/Documents/LunaNetSimulatorV2/output/ExampleEndurance/z_pr_rate.csv', names = ['Time', 'rho_rate'])
elevation_file = pd.read_csv('/Users/marta/Documents/LunaNetSimulatorV2/output/ExampleEndurance/elevation.csv', names = ['Time', 'elev'])

: 

In [ ]:
#implement measurement model
#calculate the entire Jacobian matrix
def jacobian_calc(r_tx, r_rover):
    meas_num = r_tx.shape[0]
    jac_mat = np.zeros((meas_num, 3))
    #r_tx is going to be a meas x 3 matrix and so will r_rover
    for i in range(meas_num):
        rho = r_rover-r_tx[i,:]
        rho_norm = np.linalg.norm(rho)
        jac_mat[i,:] = (rho/rho_norm).flatten()
            
    return jac_mat

#get the vector of predicted measurements
def pred_meas(r_tx, r_rover_pred):
    meas_num = r_tx.shape[0]
    delta_rho = np.zeros((meas_num, 1))
    
    for i in range(meas_num):
        rho = r_rover_pred-r_tx[i,:]
        delta_rho[i,0] = np.linalg.norm(rho)
        
    return delta_rho


: 

In [ ]:
time = rv_file['Time'].to_numpy()
################################################################################

# visible_time = time
visible_time = pseudorange_file['Time'].to_numpy()

################################################################################
r_rover = rover_file[['x', 'y', 'z']].to_numpy()
r_tx = rv_file[['x', 'y', 'z']].to_numpy()
rho_err = np.random.multivariate_normal(np.zeros(len(visible_time)), (5e-3)*np.eye(len(visible_time)))
################################################################################

indices = [ind for ind,ele in enumerate(time) if ele in visible_time]
rho_true =  pred_meas(r_tx, r_rover[0,:]).flatten()
rho_true = rho_true[indices]
sim_rho_true = pseudorange_file['rho'].to_numpy()

#################################################################################
true_meas = rho_true + rho_err
# true_meas = pseudorange_file['rho'].to_numpy()

r_est_k = np.random.multivariate_normal(r_rover[0,:], 0.01*np.eye(3))

# r_est_k = np.array([-0.1366, 0.2135, -1.7381e3])

sat_pos = np.zeros((len(visible_time), 3))
indices = [ind for ind,ele in enumerate(time) if ele in visible_time]
#get all the satellite postions at visible times
for i in range(len(indices)):
    sat_pos[i,:] = r_tx[indices[i]]


: 

In [ ]:
error_pos = np.linalg.norm((r_est_k.flatten()  - r_rover[0,:])*1000)         #m
print(f'Initial error {error_pos} m')

: 

In [ ]:
count_meas = 0
error_time = []
error_list = []
pdop_list = []
xdop_list = []
ydop_list = []
zdop_list = []

for i in range(len(time)):
    t = time[i]
    if t in visible_time:
        if count_meas > 21: #and t/3600 > 7:
            iter_count = 0
            #stack the measurements
            true_meas_stack = true_meas[0:count_meas].reshape(count_meas,1)
            #stack the satellite positions
            sat_pos_stack = sat_pos[0:count_meas, :]

            while iter_count < 1e5:
                #get the Jacobian
                jac_mat =  jacobian_calc(sat_pos_stack, r_est_k.flatten())
                # get the stack of predicted measurements
                pred_meas_stack = pred_meas(sat_pos_stack, r_est_k.flatten()).reshape(count_meas,1)
                # get residual
                dy = true_meas_stack - pred_meas_stack
                # get estiamte update
                g_mat = np.linalg.inv(jac_mat.T @ jac_mat)
                dx = g_mat @ jac_mat.T @ dy
                # update estimate
                r_est_k = r_est_k + dx.flatten()
                iter_count+=1
                # set up for new loop
                # r_est_k = r_est_kp1
                # check convergence
                if np.linalg.norm(dx) < 1e-9:
                    break
            pdop = np.sqrt(np.sum(np.diag(g_mat)))
            pdop_list.append(pdop)
            xdop_list.append(np.sqrt(np.diag(g_mat)[0]))
            ydop_list.append(np.sqrt(np.diag(g_mat)[1]))
            zdop_list.append(np.sqrt(np.diag(g_mat)[2]))

            error_pos = np.linalg.norm((r_est_k.flatten()  - r_rover[0,:])*1000)         #m
            error_list.append(error_pos)
            error_time.append(t)
            count_meas +=1
        else:
            error_pos = np.linalg.norm((r_est_k.flatten() - r_rover[0,:])*1000)         #m
            # error_list.append(error_pos)
            count_meas +=1
    else:
        error_pos = np.linalg.norm((r_est_k.flatten()  - r_rover[0,:])*1000)         #m
        # error_list.append(error_pos)

: 

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.plot(np.array(error_time)/3600, pdop_list, '.-')
plt.grid()
plt.xlabel("Measurement Time [hrs]")
plt.ylabel("PDOP")
plt.ylim(-1,10)

: 

In [ ]:
pdop_list[100]

: 

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.plot(np.array(error_time)/3600, xdop_list, '.-')
plt.plot(np.array(error_time)/3600, ydop_list, '.-')
plt.plot(np.array(error_time)/3600, zdop_list, '.-')
plt.grid()
plt.xlabel("Measurement Time [hrs]")
plt.ylabel("PDOP")


: 

In [ ]:
pdop_list[-1]

: 

In [ ]:
ydop_list[-1]

: 

In [ ]:
zdop_list[-1]

: 

In [ ]:
fig = plt.figure(figsize=(8,6))#, dpi= 500)
plt.plot(np.array(error_time)/3600, error_list, '.-')
plt.yscale('log')
plt.grid()
plt.xlabel("Measurement Time [hrs]")
plt.ylabel("Positioning Error [m]")

: 

In [ ]:
error_list[-1]

: 

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.scatter(visible_time/3600, sat_pos[:,0])
plt.scatter(visible_time/3600, sat_pos[:,1])
plt.scatter(visible_time/3600, sat_pos[:,2])
plt.xlabel("Measurement Time [hrs]")


: 

In [ ]:
fig = plt.figure(figsize=(8,6))
plt.scatter(visible_time/3600, sat_pos[:,0])
plt.xlabel("Measurement Time [hrs]")
plt.ylabel("Pseudorange [km]")

: 